# DEMO — `brz_01_arancione_sales` rewritten with the `step_log` context manager

**Illustrative only.** Paths and helper names are placeholders; this is not meant
to run. It exists to show what the proposed Decision 5 / §7 pattern looks like in
a real notebook, versus the original. Delete after reading.

## The contrast

**Original** (`vinoworld/.../brz_01_arancione_sales.ipynb`): cells 4, 5, and 6 each
end with the **same ~12-line `except` block** — `capture_exception` → build
`error_message` → `ended_timestamp` → `status = STATUS_FAILED` →
`pipeline_step_log_upsert(...)` → `raise`. Three near-identical copies in one
notebook, and the variables they touch (`status`, `ended_timestamp`,
`error_message`, `rows_read`, `rows_written`) must be pre-declared *outside* the
try blocks (CLAUDE.md §11.4) or the handler `NameError`s. That footgun exists
*because* of the copy-paste.

**Proposed:** one `with step_log(...) as step:` block wraps validate → read →
write. The context manager owns the open (`RUNNING`) and the close
(`SUCCEEDED` / `FAILED` / a caller-set terminal status like `NO_FILES`). The
notebook body has **zero** `except` boilerplate and **no** pre-declared logging
variables. You just set `step.rows_read` / `step.rows_written`.

## What `step_log` does internally (sketch — lives ONCE in `pipeline_logging.py`)

This is the whole point: the ~12-line handler is written **one time**, here,
instead of copy-pasted into every cell of every notebook.

```python
from contextlib import contextmanager

@contextmanager
def step_log(spark, audit_schema, dbutils, *, pipeline_run_id, step_sequence,
             notebook_folder, notebook_name, layer=None, target_table=None):
    state = _StepState(                       # small mutable handle the body updates
        step_log_id = str(uuid.uuid4()),
        started     = datetime.now(timezone.utc),
        status      = STATUS_RUNNING,
        rows_read   = 0,
        rows_written= 0,
    )

    # OPEN: write the RUNNING row (replaces the old "cell 3")
    pipeline_step_log_upsert(
        spark, audit_schema, state.step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, STATUS_RUNNING, state.started,
        layer, target_table)

    try:
        yield state                           # <-- the notebook body runs here

    except dbutils.NotebookExit:              # clean dbutils.notebook.exit(...)
        # caller already set state.status (e.g. STATUS_NO_FILES); log it, re-raise
        _close(spark, audit_schema, state, target_table=target_table)
        raise

    except Exception as e:                    # any real failure
        err = Utils.capture_exception(e)
        state.status = STATUS_FAILED
        _close(spark, audit_schema, state, target_table=target_table,
               error_message=f"{err['error_type']}: {err['error_message']}")
        raise

    else:                                     # body finished cleanly
        state.status = STATUS_SUCCEEDED
        _close(spark, audit_schema, state, target_table=target_table)
```

`_close(...)` just calls `pipeline_step_log_upsert(... state.status ...,
ended_timestamp=now, rows_read=state.rows_read, rows_written=state.rows_written)`.
Because the open used the same `step_log_id`, the close is a MERGE update on that
key — exactly the two-call upsert the original did by hand.

In [ ]:
%run "../../libs/notebook_init"
# (illustrative path) notebook_init injects: CATALOG, BRONZE, AUDIT, RAW_FILES,
# STATUS_RUNNING/SUCCEEDED/FAILED/NO_FILES, PIPELINE_RUN_ID, Utils, F, datetime,
# uuid, and now `step_log` + the audit helpers — all re-exported from pipeline_logging.

In [ ]:
# =============================================================================
# Constants  (UNCHANGED from the original — shown for completeness)
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

STORE_NAME     = "Arancione"
SOURCE_SUBPATH = "arancione/"
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{BRONZE}.sales_arancione"
SOURCE_ENCODING = "windows-1252"

EXPECTED_SOURCE_COLS = [
    "OnlineRetailer", "SalesMonth", "Title", "Vintage",
    "Variety", "Score", "ListPrice", "Quantity",
]
HASH_SOURCE_COLS = [
    "online_retailer", "sales_month", "title", "vintage",
    "variety", "score", "list_price", "quantity",
]
read_schema = StructType([
    StructField("OnlineRetailer", StringType(), True),
    StructField("SalesMonth",     StringType(), True),
    StructField("Title",          StringType(), True),
    StructField("Vintage",        StringType(), True),
    StructField("Variety",        StringType(), True),
    StructField("Score",          StringType(), True),
    StructField("ListPrice",      StringType(), True),
    StructField("Quantity",       StringType(), True),
])

In [ ]:
# =============================================================================
# THE WHOLE STEP — one managed block replaces original cells 3, 4, 5, 6.
#
# No try/except in the notebook. No pre-declared status/ended_timestamp/
# error_message. The context manager logs RUNNING on enter and
# SUCCEEDED / FAILED / NO_FILES on exit. You only set step.rows_* .
# =============================================================================
nb = Utils.get_notebook_context(dbutils)

with step_log(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = 1,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
) as step:

    # ---- validate: per-file header check --------------------------------
    files = [f.path for f in dbutils.fs.ls(SOURCE_PATH)
             if f.path.lower().endswith(".csv")]

    if not files:
        # Clean terminal exit. Tell the context manager the terminal status,
        # then exit — it logs NO_FILES (not FAILED) and re-raises cleanly.
        step.status = STATUS_NO_FILES
        dbutils.notebook.exit("No CSV files found at " + SOURCE_PATH)

    bad_files = []
    for file_path in files:
        actual = (spark.read.format("csv").option("header", "true")
                  .option("encoding", SOURCE_ENCODING)
                  .load(file_path).limit(0).columns)
        if actual != EXPECTED_SOURCE_COLS:
            bad_files.append((file_path, actual))
    if bad_files:
        # Just raise. The context manager captures the traceback, writes
        # FAILED to pipeline_step_log, and re-raises for the orchestrator.
        raise ValueError(f"[{TARGET_TABLE}] Header mismatch in {len(bad_files)} file(s).")

    # ---- read & shape ----------------------------------------------------
    raw_df = (spark.read.format("csv").option("header", "true")
              .option("delimiter", ",").option("encoding", SOURCE_ENCODING)
              .schema(read_schema).load(SOURCE_PATH)
              .withColumn("source_file_path", F.col("_metadata.file_path")))

    bronze_df = (
        raw_df.select(
            F.col("OnlineRetailer").alias("online_retailer"),
            F.col("SalesMonth").alias("sales_month"),
            F.col("Title").alias("title"),
            F.col("Vintage").alias("vintage"),
            F.col("Variety").alias("variety"),
            F.col("Score").alias("score"),
            F.col("ListPrice").alias("list_price"),
            F.col("Quantity").alias("quantity"),
            F.col("source_file_path"),
        )
        .withColumn("row_hash", F.md5(F.concat_ws("|", *[F.col(c) for c in HASH_SOURCE_COLS])))
        .withColumn("inserted_ts", F.current_timestamp())
        .withColumn("run_id",      F.lit(PIPELINE_RUN_ID))
        .withColumn("store_name",  F.lit(STORE_NAME))
    )

    step.rows_read = bronze_df.count()        # <-- set on the state handle

    # ---- idempotent MERGE write -----------------------------------------
    pre_count = spark.table(TARGET_TABLE).count()
    bronze_df.createOrReplaceTempView("bronze_staging")
    spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS target
        USING bronze_staging AS source
        ON target.row_hash = source.row_hash
        WHEN NOT MATCHED THEN INSERT *
    """)
    step.rows_written = spark.table(TARGET_TABLE).count() - pre_count   # <-- set on state

    # ---- leaf write: returns a dict, NEVER aborts the committed write ----
    # (Decision 4: insert-only leaf writes swallow + return {"status",...})
    res = ingestion_log_insert(
        spark, AUDIT,
        bronze_df.select("source_file_path").distinct(),
        PIPELINE_RUN_ID, step.step_log_id, STORE_NAME, TARGET_TABLE,
    )
    if res["status"] != STATUS_SUCCEEDED:
        print(f"[{TARGET_TABLE}] WARNING: ingestion_log failed, Bronze write stands. {res['error_message']}")

    # ---- archive processed files ----------------------------------------
    Utils.move_all_files(
        dbutils, source_path=SOURCE_PATH,
        target_path=f"{SOURCE_PATH}archive", create_target=True, skip_dirs=True,
    )

# On a clean exit the context manager has already written STATUS_SUCCEEDED
# with step.rows_read / step.rows_written. Nothing more to do.

## What changed vs the original

| Aspect | Original | Proposed |
|---|---|---|
| `except` handlers in the notebook | 3 copies (cells 4, 5, 6), ~12 lines each | **0** — the context manager owns it |
| Pre-declared logging vars (§11.4 footgun) | required (`status`, `ended_timestamp`, `error_message`, `rows_read`, `rows_written`) | **none** — they live on `step` |
| `pipeline_step_log_upsert` calls in the notebook | 5 (1 RUNNING + 4 terminal) | **0** — open/close happen inside `step_log` |
| Where the RUNNING/terminal logic lives | copy-pasted per notebook | **once**, in `pipeline_logging.step_log` |
| Leaf-write failure (`ingestion_log`) | `raise` (its own try/except) | returns a dict; print a warning, write still stands (Decision 4) |
| `audit_schema` | implicit via `configure()` global | explicit: `AUDIT` passed to `step_log` (Decision 1) |

The notebook now reads as a linear story (validate → read → write → log → archive)
with the failure-logging concern factored out — which is the whole proposal.